# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset ([Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)) using the `mlcroissant` library.

### Dataset Source
The dataset source is a Croissant schema accessible at the URL above. All dataset entities (record sets, fields, etc.) are referenced by their semantic `@id` fields as per the Croissant specification.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access the metadata as an object (use attributes, not dict subscripting)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets and their fields, referencing their `@id`. We'll print out available record sets, their fields, and columns.

In [ ]:
# List all record sets with their @id and names, and fields within each.
record_set_objs = list(dataset.metadata.record_sets)
print(f"Number of record sets: {len(record_set_objs)}\n")

for rs in record_set_objs:
    print(f"Record Set @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {rs.description}")
    if hasattr(rs, 'fields') and rs.fields is not None:
        print("  Fields:")
        for field in rs.fields:
            print(f"    Field @id: {field.id} | Name: {field.name} | Description: {field.description}")
    if hasattr(rs, 'columns') and rs.columns is not None:
        print("  Columns:")
        for col in rs.columns:
            print(f"    Column @id: {col.id} | Name: {col.name} | Description: {col.description}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview above.

Below, we extract **all** record sets as an example. Replace the list below with the specific `@id` of the record set you're interested in for a targeted workflow.

In [ ]:
# Extract data from each record set
record_sets = [rs.id for rs in record_set_objs]
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    if len(df) > 0:
        print(f"  Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"  Record set {record_set_id} has no records.")

# Choose the primary record set for further analysis (update as needed from the printed record sets)
primary_record_set_id = record_sets[0] if record_sets else None
if primary_record_set_id is not None:
    print(f"Selected primary record set: {primary_record_set_id}")
    print(f"Columns: {dataframes[primary_record_set_id].columns.tolist()}")
    display(dataframes[primary_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps—filter numeric fields, normalize, group, etc. All fields/columns are referenced by their `@id`. **Update the `numeric_field_id` and `group_field_id` below to match the identifiers printed earlier**.

In [ ]:
# EXAMPLE: Specify field @ids relevant to the dataset.
# Replace with actual field/column @id from record set (see output above, e.g. 'cr:abundance', 'cr:MW')
numeric_field_id = None  # Example: 'cr:MW', update as needed
group_field_id = None    # Example: 'cr:SampleGroup', update as needed

df = dataframes[primary_record_set_id]
if numeric_field_id is not None and numeric_field_id in df.columns:
    # Filter for values > threshold
    threshold = 10
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id is not None and group_field_id in df.columns:
        grouped_df = (
            filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        )
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df)
else:
    print("Please set `numeric_field_id` and `group_field_id` to appropriate field/column @id from the record set.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Replace the field @ids in the plot below based on your data exploration above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example: Histogram of numeric field
if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
else:
    print("Set `numeric_field_id` to a valid numeric @id to see histogram.")

# Example: Boxplot by group if possible
if numeric_field_id is not None and group_field_id is not None and group_field_id in df.columns:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset using the `mlcroissant` library, demonstrated how to reference and analyze record sets and fields by their `@id`, and outlined sample EDA and visualization steps. Adjust the field and group selections above as appropriate to your research focus.